# Taller 4 

**Asignatura:** Aprendizaje Computacional  
**Programa:** Ingeniería de Sistemas  
**GRUPO:** Robin Alexander Montoya, Tomas Correa y Samuel Arango

## Dataset 
link: https://www.kaggle.com/datasets/arpitsharmaml/student-academic-risk-prediction-dataset/data#1

Se utiliza el archivo **student_data_100000.csv**, ubicado en `archive (2)`. Contiene información sobre asistencia, horas de estudio, desempeño académico y contexto del estudiante.

**Objetivo:** clasificar el nivel de riesgo académico (`risk_category`) en tres categorías: `Safe`, `At-Risk` o `High-Risk`.

## 1.2 Preprocesamiento

Carga del dataset, verificación de valores faltantes y nulos, limpieza y definición de los pipelines de preprocesamiento e imputación.

In [2]:
from pathlib import Path
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Carga de datos
DATA_PATH = Path("archive (2)") / "student_data_100000.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(f"No se encontró el dataset: {DATA_PATH.resolve()}")

df = pd.read_csv(DATA_PATH)
TARGET = "risk_category"
if TARGET not in df.columns:
    raise ValueError(f"No se encontró la columna objetivo '{TARGET}' en el dataset.")

# Verificación de calidad de datos
print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")
print(f"Valores faltantes: {df.isna().sum().sum()}")
print(f"Filas duplicadas: {df.duplicated().sum()}")

# Definición de transformadores y pipelines de preprocesamiento
feature_columns = df.drop(columns=[TARGET]).columns
numeric_features = df[feature_columns].select_dtypes(include="number").columns.tolist()
categorical_features = df[feature_columns].select_dtypes(exclude="number").columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_features,
        ),
        (
            "categorical",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_features,
        )
    ],
    remainder="drop",
)
print("Preprocesamiento configurado correctamente.")

Filas: 100,000 | Columnas: 11
Valores faltantes: 2000
Filas duplicadas: 0
Preprocesamiento configurado correctamente.


## 1.3 Definir X e Y, describir columnas y .head()

Se separan las variables características ($X$) de la variable objetivo ($Y$). Se muestra la descripción de los datos y las primeras filas del dataset.

In [3]:
# Definición de X e Y
X = df.drop(columns=[TARGET]).copy()
y = df[TARGET].copy()

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

# Descripción de columnas y tipos de datos
print("\n--- Tipos de datos y no nulos ---")
display(pd.DataFrame({"tipo": X.dtypes.astype(str), "no_nulos": X.notna().sum()}))

# Estadísticas descriptivas de las columnas
print("\n--- Estadísticas descriptivas ---")
display(X.describe().T)

# Primeras filas (.head())
print("\n--- Primeras 5 filas (.head()) ---")
display(df.head())

X shape: (100000, 10)
y shape: (100000,)

--- Tipos de datos y no nulos ---


,tipo,no_nulos
attendance,float64,99000
study_hours,float64,99000
past_failures,int64,100000
assignments_completed_pct,float64,100000
parental_education,str,100000
family_income,str,100000
extracurricular,str,100000
internet_access,str,100000
previous_grade,float64,100000
final_score,float64,100000



--- Estadísticas descriptivas ---


,count,mean,std,min,25%,50%,75%,max
attendance,99000.0,72.972567,21.562735,10.0,58.15,75.065,91.94,100.0
study_hours,99000.0,6.101901,3.722512,0.0,3.30,6.000,8.70,15.0
past_failures,100000.0,1.848650,1.491297,0.0,1.00,2.000,3.00,5.0
assignments_completed_pct,100000.0,72.929616,21.656246,0.0,58.09,75.070,91.83,100.0
previous_grade,100000.0,72.967067,21.624269,0.0,58.19,75.040,91.89,100.0
final_score,100000.0,62.968236,15.888285,0.0,52.22,63.160,73.99,100.0



--- Primeras 5 filas (.head()) ---


,attendance,study_hours,past_failures,assignments_completed_pct,parental_education,family_income,extracurricular,internet_access,previous_grade,final_score,risk_category
0,87.42,10.12,2,100.00,Secondary,Low,Yes,Yes,46.80,76.08,Safe
1,71.54,1.38,3,36.70,Secondary,High,Yes,Yes,48.05,43.21,High-Risk
2,91.19,8.30,5,64.38,Secondary,Low,No,Yes,69.40,60.89,At-Risk
3,100.00,3.52,4,93.77,Secondary,Low,No,No,58.07,45.47,At-Risk
4,69.15,4.69,1,72.36,Primary,Low,Yes,Yes,50.55,37.15,High-Risk


## 1.4 Train/test split

Se realiza la división estratificada del conjunto de datos en entrenamiento (80%) y prueba (20%), conservando la proporción de las tres categorías de riesgo.

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Entrenamiento (Train): {X_train.shape[0]} filas")
print(f"Prueba (Test): {X_test.shape[0]} filas")

print("\nDistribución de clases en Entrenamiento:")
print(y_train.value_counts())

print("\nDistribución de clases en Prueba:")
print(y_test.value_counts())

Entrenamiento (Train): 80000 filas
Prueba (Test): 20000 filas

Distribución de clases en Entrenamiento:
risk_category
At-Risk      39992
Safe         20009
High-Risk    19999
Name: count, dtype: int64

Distribución de clases en Prueba:
risk_category
At-Risk      9998
Safe         5002
High-Risk    5000
Name: count, dtype: int64


## 1.5 Entrenamiento y evaluación del modelo KNN

Se ajusta el preprocesamiento solo con los datos de entrenamiento y se evalúa el clasificador con el conjunto de prueba. Al trabajar con 100.000 registros, la predicción puede tardar según el equipo.

In [5]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier

knn_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", KNeighborsClassifier(n_neighbors=5, n_jobs=-1)),
    ]
)

knn_model.fit(X_train, y_train)
y_pred = knn_model.predict(X_test)

print(f"Exactitud (accuracy): {accuracy_score(y_test, y_pred):.4f}")
print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred, zero_division=0))

labels = knn_model.named_steps["classifier"].classes_
confusion = confusion_matrix(y_test, y_pred, labels=labels)
display(pd.DataFrame(confusion, index=labels, columns=labels).rename_axis("Real").rename_axis("Predicción", axis=1))

Exactitud (accuracy): 0.8722

Reporte de clasificación:
              precision    recall  f1-score   support

     At-Risk       0.85      0.90      0.88      9998
   High-Risk       0.90      0.83      0.86      5000
        Safe       0.89      0.86      0.87      5002

    accuracy                           0.87     20000
   macro avg       0.88      0.86      0.87     20000
weighted avg       0.87      0.87      0.87     20000



Predicción,At-Risk,High-Risk,Safe
Real,,,
At-Risk,9003,451,544
High-Risk,845,4155,0
Safe,713,2,4287
